In [11]:
import os
import mesa
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

def get_live_market_price(commodity: str) -> float:
    """Fetches the current real-time market price for a given commodity."""
    print(f"⚙️ [SYSTEM] Python hardware triggered natively for commodity: '{commodity}'")
    
    prices = {"gold": 2500.50, "silver": 30.25, "copper": 4.15, "nexus_core": 9450.75}
    return prices.get(commodity.lower(), 0.0)

class ToolCallingAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)

    def step(self):
        print(f"\n--- Agent {self.unique_id} Step Initiated ---")
        
        tool_config = types.GenerateContentConfig(
            tools=[get_live_market_price],
            temperature=0.0
        )
        
        chat = client.chats.create(model='gemini-2.5-flash', config=tool_config)
        
        prompt = """
        You are a ruthless trader. You MUST fetch the exact market price of the commodity 'nexus_core'.
        You have zero prior knowledge of what a 'nexus_core' is.
        If the price is over 5000, respond: "The price is [EXACT_NUMBER]. Too expensive, I pass."
        If the price is under 5000, respond: "The price is [EXACT_NUMBER]. I am buying everything."
        """
        
        try:
            response = chat.send_message(prompt)
            
            print(f"🤖 Agent {self.unique_id} Final Decision: {response.text}")

        except Exception as e:
            print(f"Error: {e}")
            
        time.sleep(4)

class ToolModel(mesa.Model):
    def __init__(self):
        super().__init__()
        self.agent = ToolCallingAgent(self)
        
    def step(self):
        self.agent.step()


--- Agent 1 Step Initiated ---
⚙️ [SYSTEM] Python hardware triggered natively for commodity: 'nexus_core'
🤖 Agent 1 Final Decision: The price is 9450.75. Too expensive, I pass.


In [12]:
model = ToolModel()
model.step()


--- Agent 1 Step Initiated ---
⚙️ [SYSTEM] Python hardware triggered natively for commodity: 'nexus_core'
🤖 Agent 1 Final Decision: The price is 9450.75. Too expensive, I pass.
